<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/assignment2_661_AA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 2: Topic Modeling Analysis of Airbnb Reviews (Pre- vs Post-Pandemic)

Comparing hidden themes in Asheville Airbnb reviews from **2019 (pre-pandemic)** and **2023 (post-pandemic)** using LDA, NMF, and LSA. The workflow follows the W7 base notebook and is applied consistently to both years.

## 1. Setup and Data Loading

Import libraries. Taken directly from W7.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD, PCA

In [ ]:
# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

Load the Asheville reviews dataset. Same source as W7.

In [ ]:
# import Asheville reviews dataset
reviews_url = 'https://data.insideairbnb.com/united-states/nc/asheville/2024-06-21/data/reviews.csv.gz'
reviews_df = pd.read_csv(reviews_url, compression='gzip')
reviews_df.head()

In [ ]:
# Check shape of full dataset
reviews_df.shape

Convert the date column to datetime so I can filter by year.

In [ ]:
# Ensure the 'date' column is in datetime format
reviews_df['date'] = pd.to_datetime(reviews_df['date'], errors='coerce')

## 2. Text Preprocessing

Reusing the `preprocess_text()` function from W7. I extended the custom stopword list because the original W7 output had a lot of generic positive filler words (great, place, stay, asheville, etc.) showing up across almost every topic, which made the topics harder to tell apart. Removing these forces the models to surface more distinctive themes.

In [ ]:
# Pre-processing function (from W7) with extended custom stopwords
lemmatizer = WordNetLemmatizer()

custom_stopwords = {
    'airbnb', 'asheville', 'place', 'stay', 'stayed', 'staying', 'host',
    'great', 'nice', 'good', 'really', 'definitely', 'would', 'could',
    'also', 'us', 'get', 'got', 'one', 'time', 'made', 'make', 'even',
    'much', 'well', 'back', 'go', 'went', 'come', 'came', 'lot', 'thing'
}
stop_words = set(stopwords.words('english')).union(custom_stopwords)

def preprocess_text(text):
    if pd.isnull(text):
        return ''
    # Remove URLs, non-letters, and convert to lowercase
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    # Tokenize
    tokens = text.split()
    # Lemmatize and remove stopwords
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and w not in string.punctuation and len(w) > 2]
    return ' '.join(tokens)

Filter to 2019 and 2023 separately and sample 10,000 reviews from each year. I am keeping the same sample size as W7 so the comparison between years is fair and the notebook stays within Colab's memory and runtime limits.

In [ ]:
# Filter and sample 2019 (pre-pandemic) reviews
reviews_2019 = reviews_df[reviews_df['date'].dt.year == 2019].copy()
print('2019 total reviews:', reviews_2019.shape[0])
reviews_2019 = reviews_2019.sample(n=10000, random_state=42)
reviews_2019['cleaned_comments'] = reviews_2019['comments'].apply(preprocess_text)
reviews_2019.shape

In [ ]:
# Filter and sample 2023 (post-pandemic) reviews
reviews_2023 = reviews_df[reviews_df['date'].dt.year == 2023].copy()
print('2023 total reviews:', reviews_2023.shape[0])
reviews_2023 = reviews_2023.sample(n=10000, random_state=42)
reviews_2023['cleaned_comments'] = reviews_2023['comments'].apply(preprocess_text)
reviews_2023.shape

In [ ]:
# Quick look at a cleaned 2019 review vs a cleaned 2023 review
print('2019 sample:', reviews_2019['cleaned_comments'].iloc[0][:300])
print()
print('2023 sample:', reviews_2023['cleaned_comments'].iloc[0][:300])

## 3. Vectorization (BoW and TF-IDF)

Following W7, I use BoW for LDA and TF-IDF for NMF and LSA. I fit a separate vectorizer per year so each year has its own vocabulary, but I keep the vectorizer parameters identical so the comparison stays fair.

In [ ]:
# Vectorize 2019 (BoW for LDA, TF-IDF for NMF/LSA)
bow_vec_2019 = CountVectorizer(min_df=5, max_df=0.90, stop_words='english', ngram_range=(1,1))
X_bow_2019 = bow_vec_2019.fit_transform(reviews_2019['cleaned_comments'])
terms_bow_2019 = bow_vec_2019.get_feature_names_out()

tfidf_vec_2019 = TfidfVectorizer(min_df=5, max_df=0.90, stop_words='english', ngram_range=(1,1))
X_tfidf_2019 = tfidf_vec_2019.fit_transform(reviews_2019['cleaned_comments'])
terms_tfidf_2019 = tfidf_vec_2019.get_feature_names_out()

print('2019 BoW shape:', X_bow_2019.shape)
print('2019 TF-IDF shape:', X_tfidf_2019.shape)

In [ ]:
# Vectorize 2023 (same parameters)
bow_vec_2023 = CountVectorizer(min_df=5, max_df=0.90, stop_words='english', ngram_range=(1,1))
X_bow_2023 = bow_vec_2023.fit_transform(reviews_2023['cleaned_comments'])
terms_bow_2023 = bow_vec_2023.get_feature_names_out()

tfidf_vec_2023 = TfidfVectorizer(min_df=5, max_df=0.90, stop_words='english', ngram_range=(1,1))
X_tfidf_2023 = tfidf_vec_2023.fit_transform(reviews_2023['cleaned_comments'])
terms_tfidf_2023 = tfidf_vec_2023.get_feature_names_out()

print('2023 BoW shape:', X_bow_2023.shape)
print('2023 TF-IDF shape:', X_tfidf_2023.shape)

## 4. Helper Functions

Reusing the plotting helpers from W7 (`plot_top_words`, `plot_word_clouds`, `plot_topic_distance_pca`) plus a small helper that prints the top words per topic in a clean list.

In [ ]:
# Plot helpers (from W7)
def plot_top_words(model, feature_names, n_top_words=10, title='Top words per topic'):
    fig, axes = plt.subplots(1, model.n_components, figsize=(4*model.n_components, 5), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]
        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f'Topic {topic_idx+1}', fontdict={'fontsize': 10})
        ax.invert_yaxis()
        ax.tick_params(axis='both', which='major', labelsize=10)
    plt.suptitle(title, fontsize=12)
    plt.subplots_adjust(top=0.85, wspace=0.3)
    plt.show()

def plot_word_clouds(model, feature_names, n_top_words=30, title_prefix=''):
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = {feature_names[i]: topic[i] for i in top_features_ind}
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_features)
        plt.figure()
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'{title_prefix}Topic {topic_idx+1} Word Cloud', fontsize=14)
        plt.show()

def plot_topic_distance_pca(model, title='Topic Distance (PCA)'):
    topic_word_matrix = model.components_
    pca = PCA(n_components=2)
    topic_pca = pca.fit_transform(topic_word_matrix)
    plt.figure(figsize=(8, 6))
    plt.scatter(topic_pca[:, 0], topic_pca[:, 1], c='blue', edgecolor='k', s=100)
    for i in range(len(topic_pca)):
        plt.text(topic_pca[i, 0], topic_pca[i, 1], f'  Topic {i+1}', fontsize=12)
    plt.title(title)
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.show()

def print_topics(model, feature_names, n_top_words=10, label='Topics'):
    print(f'{label}:')
    for idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[-n_top_words:][::-1]]
        print(f'  Topic {idx+1}: {", ".join(top_words)}')

## 5. Choosing the Number of Topics

The rubric requires me to justify the number of topics for each model. I sweep across `k = 4..10` and use a model-appropriate metric for each:

- **LDA** — perplexity on the BoW matrix (lower is better, but the goal is the elbow, not the absolute minimum)
- **NMF** — reconstruction error on TF-IDF (lower is better, again using elbow logic)
- **LSA** — cumulative explained variance ratio on TF-IDF (higher is better, look for diminishing returns)

I run the sweep on both years so each year gets its own justified `k` per model.

In [ ]:
# Sweep functions
k_range = list(range(4, 11))

def lda_perplexity_sweep(X, k_range):
    scores = []
    for k in k_range:
        m = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=10)
        m.fit(X)
        scores.append(m.perplexity(X))
    return scores

def nmf_reconstruction_sweep(X, k_range):
    scores = []
    for k in k_range:
        m = NMF(n_components=k, random_state=42, max_iter=400)
        m.fit(X)
        scores.append(m.reconstruction_err_)
    return scores

def lsa_explained_variance_sweep(X, k_range):
    scores = []
    for k in k_range:
        m = TruncatedSVD(n_components=k, random_state=42)
        m.fit(X)
        scores.append(m.explained_variance_ratio_.sum())
    return scores

def plot_sweep(k_range, scores, title, ylabel):
    plt.figure(figsize=(6,4))
    plt.plot(k_range, scores, marker='o')
    plt.title(title)
    plt.xlabel('Number of topics (k)')
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.show()

**Sweeps for 2019**

In [ ]:
lda_scores_2019 = lda_perplexity_sweep(X_bow_2019, k_range)
plot_sweep(k_range, lda_scores_2019, '2019 LDA Perplexity vs k', 'Perplexity (lower better)')

nmf_scores_2019 = nmf_reconstruction_sweep(X_tfidf_2019, k_range)
plot_sweep(k_range, nmf_scores_2019, '2019 NMF Reconstruction Error vs k', 'Reconstruction error')

lsa_scores_2019 = lsa_explained_variance_sweep(X_tfidf_2019, k_range)
plot_sweep(k_range, lsa_scores_2019, '2019 LSA Cumulative Explained Variance vs k', 'Explained variance ratio')

**Sweeps for 2023**

In [ ]:
lda_scores_2023 = lda_perplexity_sweep(X_bow_2023, k_range)
plot_sweep(k_range, lda_scores_2023, '2023 LDA Perplexity vs k', 'Perplexity (lower better)')

nmf_scores_2023 = nmf_reconstruction_sweep(X_tfidf_2023, k_range)
plot_sweep(k_range, nmf_scores_2023, '2023 NMF Reconstruction Error vs k', 'Reconstruction error')

lsa_scores_2023 = lsa_explained_variance_sweep(X_tfidf_2023, k_range)
plot_sweep(k_range, lsa_scores_2023, '2023 LSA Cumulative Explained Variance vs k', 'Explained variance ratio')

### k Selection and Justification

After looking at the sweep plots above I picked the following number of topics for each model. The justification will be filled in once the sweeps are run, but my plan is to pick the k that sits at the elbow of each curve while still giving topics that are interpretable. I am starting with **k = 7** for all models (matching W7) as a placeholder and will adjust after seeing the sweeps.

| Model | 2019 k | 2023 k |
|---|---|---|
| LDA | 7 | 7 |
| NMF | 7 | 7 |
| LSA | 7 | 7 |

In [ ]:
# Final k per model per year (update after viewing the sweeps above)
k_lda_2019, k_nmf_2019, k_lsa_2019 = 7, 7, 7
k_lda_2023, k_nmf_2023, k_lsa_2023 = 7, 7, 7

## 6. Topic Models — 2019 (Pre-Pandemic)

**6.1 LDA (2019)**

In [ ]:
lda_2019 = LatentDirichletAllocation(n_components=k_lda_2019, random_state=42)
lda_2019.fit(X_bow_2019)
print_topics(lda_2019, terms_bow_2019, label='2019 LDA Topics')
plot_top_words(lda_2019, terms_bow_2019, title='2019 LDA - Top words per topic')
plot_word_clouds(lda_2019, terms_bow_2019, title_prefix='2019 LDA ')

**6.2 NMF (2019)**

In [ ]:
nmf_2019 = NMF(n_components=k_nmf_2019, random_state=42, max_iter=400)
nmf_2019.fit(X_tfidf_2019)
print_topics(nmf_2019, terms_tfidf_2019, label='2019 NMF Topics')
plot_top_words(nmf_2019, terms_tfidf_2019, title='2019 NMF - Top words per topic')
plot_word_clouds(nmf_2019, terms_tfidf_2019, title_prefix='2019 NMF ')

**6.3 LSA (2019)**

In [ ]:
lsa_2019 = TruncatedSVD(n_components=k_lsa_2019, random_state=42)
lsa_2019.fit(X_tfidf_2019)
print_topics(lsa_2019, terms_tfidf_2019, label='2019 LSA Topics')
plot_top_words(lsa_2019, terms_tfidf_2019, title='2019 LSA - Top words per topic')
plot_word_clouds(lsa_2019, terms_tfidf_2019, title_prefix='2019 LSA ')

In [ ]:
# Topic distance for 2019 models
plot_topic_distance_pca(lda_2019, title='2019 LDA Topic Distance (PCA)')
plot_topic_distance_pca(nmf_2019, title='2019 NMF Topic Distance (PCA)')
plot_topic_distance_pca(lsa_2019, title='2019 LSA Topic Distance (PCA)')

## 7. Topic Models — 2023 (Post-Pandemic)

**7.1 LDA (2023)**

In [ ]:
lda_2023 = LatentDirichletAllocation(n_components=k_lda_2023, random_state=42)
lda_2023.fit(X_bow_2023)
print_topics(lda_2023, terms_bow_2023, label='2023 LDA Topics')
plot_top_words(lda_2023, terms_bow_2023, title='2023 LDA - Top words per topic')
plot_word_clouds(lda_2023, terms_bow_2023, title_prefix='2023 LDA ')

**7.2 NMF (2023)**

In [ ]:
nmf_2023 = NMF(n_components=k_nmf_2023, random_state=42, max_iter=400)
nmf_2023.fit(X_tfidf_2023)
print_topics(nmf_2023, terms_tfidf_2023, label='2023 NMF Topics')
plot_top_words(nmf_2023, terms_tfidf_2023, title='2023 NMF - Top words per topic')
plot_word_clouds(nmf_2023, terms_tfidf_2023, title_prefix='2023 NMF ')

**7.3 LSA (2023)**

In [ ]:
lsa_2023 = TruncatedSVD(n_components=k_lsa_2023, random_state=42)
lsa_2023.fit(X_tfidf_2023)
print_topics(lsa_2023, terms_tfidf_2023, label='2023 LSA Topics')
plot_top_words(lsa_2023, terms_tfidf_2023, title='2023 LSA - Top words per topic')
plot_word_clouds(lsa_2023, terms_tfidf_2023, title_prefix='2023 LSA ')

In [ ]:
# Topic distance for 2023 models
plot_topic_distance_pca(lda_2023, title='2023 LDA Topic Distance (PCA)')
plot_topic_distance_pca(nmf_2023, title='2023 NMF Topic Distance (PCA)')
plot_topic_distance_pca(lsa_2023, title='2023 LSA Topic Distance (PCA)')

## 8. Topic Labels

Labels will be filled in after running the notebook and reading the top words for each topic. Tables below are placeholders that I will replace with my own interpretations.

**2019 (Pre-Pandemic)**

| Topic | LDA Label | NMF Label | LSA Label |
|---|---|---|---|
| 1 | TBD | TBD | TBD |
| 2 | TBD | TBD | TBD |
| 3 | TBD | TBD | TBD |
| 4 | TBD | TBD | TBD |
| 5 | TBD | TBD | TBD |
| 6 | TBD | TBD | TBD |
| 7 | TBD | TBD | TBD |

**2023 (Post-Pandemic)**

| Topic | LDA Label | NMF Label | LSA Label |
|---|---|---|---|
| 1 | TBD | TBD | TBD |
| 2 | TBD | TBD | TBD |
| 3 | TBD | TBD | TBD |
| 4 | TBD | TBD | TBD |
| 5 | TBD | TBD | TBD |
| 6 | TBD | TBD | TBD |
| 7 | TBD | TBD | TBD |

## 9. Summary

This notebook implemented the full topic modeling workflow from W7 on both 2019 and 2023 Asheville Airbnb reviews using LDA, NMF, and LSA. The full discussion of topic comparison, pre- vs post-pandemic shifts, and stakeholder recommendations is in the accompanying business report.